In [ ]:
import pandas as pd
import sqlite3
from google.colab import files

uploaded = files.upload()

Saving coffee_shop_sales_analysis.xlsx to coffee_shop_sales_analysis.xlsx


In [ ]:
df = pd.read_excel(
    "coffee_shop_sales_analysis.xlsx",
    sheet_name="Transactions"
)

df.head()

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail,revenue,hour,day_of_week,month,month_number,weekday_number
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg,6.0,7,Sunday,Jan,1,7
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,6.2,7,Sunday,Jan,1,7
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg,9.0,7,Sunday,Jan,1,7
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm,2.0,7,Sunday,Jan,1,7
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,6.2,7,Sunday,Jan,1,7


In [ ]:
conn = sqlite3.connect("coffee_shop_sales.db")

df.to_sql(
    "transactions",
    conn,
    if_exists="replace",
    index=False
)

149116

In [ ]:
pd.read_sql_query("""
SELECT COUNT(*) AS total_rows
FROM transactions;
""", conn)

,total_rows
0,149116


In [ ]:
pd.read_sql_query("""
SELECT
    COUNT(*) AS total_transactions,
    SUM(transaction_qty) AS items_sold,
    ROUND(SUM(revenue), 2) AS total_revenue,
    ROUND(SUM(revenue) / COUNT(*), 2) AS average_transaction_value
FROM transactions;
""", conn)

,total_transactions,items_sold,total_revenue,average_transaction_value
0,149116,214470,698812.33,4.69


Revenue by Store

In [ ]:
pd.read_sql_query("""
SELECT
    store_location,
    COUNT(*) AS transactions,
    SUM(transaction_qty) AS items_sold,
    ROUND(SUM(revenue), 2) AS revenue
FROM transactions
GROUP BY store_location
ORDER BY revenue DESC;
""", conn)

,store_location,transactions,items_sold,revenue
0,Hell's Kitchen,50735,71737,236511.17
1,Astoria,50599,70991,232243.91
2,Lower Manhattan,47782,71742,230057.25


Monthly Revenue Trend

In [ ]:
pd.read_sql_query("""
SELECT
    month_number,
    month,
    COUNT(*) AS transactions,
    SUM(transaction_qty) AS items_sold,
    ROUND(SUM(revenue), 2) AS revenue
FROM transactions
GROUP BY month_number, month
ORDER BY month_number;
""", conn)

,month_number,month,transactions,items_sold,revenue
0,1,Jan,17314,24870,81677.74
1,2,Feb,16359,23550,76145.19
2,3,Mar,21229,30406,98834.68
3,4,Apr,25335,36469,118941.08
4,5,May,33527,48233,156727.76
5,6,Jun,35352,50942,166485.88


Revenue by Hour

In [ ]:
pd.read_sql_query("""
SELECT
    hour,
    COUNT(*) AS transactions,
    SUM(transaction_qty) AS items_sold,
    ROUND(SUM(revenue), 2) AS revenue
FROM transactions
GROUP BY hour
ORDER BY hour;
""", conn)

,hour,transactions,items_sold,revenue
0,6,4594,6865,21900.27
1,7,13428,19449,63526.47
2,8,17654,25197,82699.87
3,9,17764,25370,85169.53
4,10,18545,26713,88673.39
5,11,9766,14035,46319.14
6,12,8708,12690,40192.79
7,13,8714,12439,40367.45
8,14,8933,12907,41304.74
9,15,8979,12923,41733.10


Revenue by Profuct Category

In [ ]:
pd.read_sql_query("""
SELECT
    product_category,
    SUM(transaction_qty) AS items_sold,
    ROUND(SUM(revenue), 2) AS revenue
FROM transactions
GROUP BY product_category
ORDER BY revenue DESC;
""", conn)

,product_category,items_sold,revenue
0,Coffee,89250,269952.45
1,Tea,69737,196405.95
2,Bakery,23214,82315.64
3,Drinking Chocolate,17457,72416.00
4,Coffee beans,1828,40085.25
5,Branded,776,13607.00
6,Loose Tea,1210,11213.60
7,Flavours,10511,8408.80
8,Packaged Chocolate,487,4407.64


Sales by Day of Week

In [ ]:
pd.read_sql_query("""
SELECT
    weekday_number,
    day_of_week,
    COUNT(*) AS transactions,
    SUM(transaction_qty) AS items_sold,
    ROUND(SUM(revenue), 2) AS revenue
FROM transactions
GROUP BY weekday_number, day_of_week
ORDER BY weekday_number;
""", conn)

,weekday_number,day_of_week,transactions,items_sold,revenue
0,1,Monday,21643,31231,101677.28
1,2,Tuesday,21202,30449,99455.94
2,3,Wednesday,21310,30625,100313.54
3,4,Thursday,21654,31162,100767.78
4,5,Friday,21701,31207,101373.00
5,6,Saturday,20510,29614,96894.48
6,7,Sunday,21096,30182,98330.31


Top 10 Product Types by Revenue

In [ ]:
pd.read_sql_query("""
SELECT
    product_category,
    product_type,
    SUM(transaction_qty) AS items_sold,
    ROUND(SUM(revenue), 2) AS revenue
FROM transactions
GROUP BY product_category, product_type
ORDER BY revenue DESC
LIMIT 10;
""", conn)

,product_category,product_type,items_sold,revenue
0,Coffee,Barista Espresso,24943,91406.20
1,Tea,Brewed Chai tea,26250,77081.95
2,Drinking Chocolate,Hot chocolate,17457,72416.00
3,Coffee,Gourmet brewed coffee,25973,70034.60
4,Tea,Brewed Black tea,17462,47932.00
5,Tea,Brewed herbal tea,17328,47539.50
6,Coffee,Premium brewed coffee,12431,38781.15
7,Coffee,Organic brewed coffee,13012,37746.50
8,Bakery,Scone,10465,36866.12
9,Coffee,Drip coffee,12891,31984.00


Highest-Revenue Hour at Each Store

In [ ]:
pd.read_sql_query("""
WITH hourly_sales AS (
    SELECT
        store_location,
        hour,
        ROUND(SUM(revenue), 2) AS revenue
    FROM transactions
    GROUP BY store_location, hour
),
ranked_hours AS (
    SELECT
        store_location,
        hour,
        revenue,
        ROW_NUMBER() OVER (
            PARTITION BY store_location
            ORDER BY revenue DESC
        ) AS revenue_rank
    FROM hourly_sales
)
SELECT
    store_location,
    hour AS highest_revenue_hour,
    revenue
FROM ranked_hours
WHERE revenue_rank = 1
ORDER BY revenue DESC;
""", conn)

,store_location,highest_revenue_hour,revenue
0,Hell's Kitchen,10,33605.81
1,Lower Manhattan,10,30641.46
2,Astoria,10,24426.12


In [ ]:
%%writefile coffee_shop_sales_analysis.sql

-- 1. Overall dataset summary
SELECT
    COUNT(*) AS total_transactions,
    SUM(transaction_qty) AS items_sold,
    ROUND(SUM(revenue), 2) AS total_revenue,
    ROUND(SUM(revenue) / COUNT(*), 2) AS average_transaction_value
FROM transactions;


-- 2. Performance by store
SELECT
    store_location,
    COUNT(*) AS transactions,
    SUM(transaction_qty) AS items_sold,
    ROUND(SUM(revenue), 2) AS revenue
FROM transactions
GROUP BY store_location
ORDER BY revenue DESC;


-- 3. Monthly sales trend
SELECT
    month_number,
    month,
    COUNT(*) AS transactions,
    SUM(transaction_qty) AS items_sold,
    ROUND(SUM(revenue), 2) AS revenue
FROM transactions
GROUP BY month_number, month
ORDER BY month_number;


-- 4. Sales by hour
SELECT
    hour,
    COUNT(*) AS transactions,
    SUM(transaction_qty) AS items_sold,
    ROUND(SUM(revenue), 2) AS revenue
FROM transactions
GROUP BY hour
ORDER BY hour;

-- 5. Revenue by product category
SELECT
    product_category,
    SUM(transaction_qty) AS items_sold,
    ROUND(SUM(revenue), 2) AS revenue
FROM transactions
GROUP BY product_category
ORDER BY revenue DESC;


-- 6. Sales by day of week
SELECT
    weekday_number,
    day_of_week,
    COUNT(*) AS transactions,
    SUM(transaction_qty) AS items_sold,
    ROUND(SUM(revenue), 2) AS revenue
FROM transactions
GROUP BY weekday_number, day_of_week
ORDER BY weekday_number;


-- 7. Top 10 product types by revenue
SELECT
    product_category,
    product_type,
    SUM(transaction_qty) AS items_sold,
    ROUND(SUM(revenue), 2) AS revenue
FROM transactions
GROUP BY product_category, product_type
ORDER BY revenue DESC
LIMIT 10;


-- 8. Highest-revenue hour at each store
WITH hourly_sales AS (
    SELECT
        store_location,
        hour,
        ROUND(SUM(revenue), 2) AS revenue
    FROM transactions
    GROUP BY store_location, hour
),
ranked_hours AS (
    SELECT
        store_location,
        hour,
        revenue,
        ROW_NUMBER() OVER (
            PARTITION BY store_location
            ORDER BY revenue DESC
        ) AS revenue_rank
    FROM hourly_sales
)
SELECT
    store_location,
    hour AS highest_revenue_hour,
    revenue
FROM ranked_hours
WHERE revenue_rank = 1
ORDER BY revenue DESC;

Overwriting coffee_shop_sales_analysis.sql


In [ ]:
files.download("coffee_shop_sales_analysis.sql")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>